# Climate Baseline v2 — Next-Hour Flooding Report Prediction

**Author:** Muhammad Kaleem — Climate Module Lead  
**Project:** Plexa — Multi-Modal Smart City Digital Twin and Cascading Resilience Engine

---

This notebook builds on the corrected climate preparation pipeline, which
combines NYC street geometry, terrain features, hourly MRMS rainfall and
qualifying NYC 311 flooding reports.

Here I will review the prepared dataset, compare baseline models and evaluate
their ability to predict whether a physical street segment receives a flooding
report during the next hour. The selected model will support the climate
module's integration with the cascading engine.

### Baseline Features (X)

| Feature | Source | Description |
|:---|:---|:---|
| `elevation_mean_m` | SRTM terrain raster | Mean sampled elevation along the street segment, in metres |
| `terrain_endpoint_grade_abs` | SRTM terrain and street geometry | Absolute endpoint elevation difference divided by segment length |
| `terrain_relief_m` | SRTM terrain raster | Maximum minus minimum sampled elevation, in metres |
| `length_m` | OSM street geometry | Physical street segment length, in metres |
| `rain_1h_mm` | NOAA MRMS radar | Rainfall accumulated over 1 consecutive hour |
| `rain_3h_mm` | NOAA MRMS radar | Rainfall accumulated over 3 consecutive hours |
| `rain_6h_mm` | NOAA MRMS radar | Rainfall accumulated over 6 consecutive hours |
| `rain_24h_mm` | NOAA MRMS radar | Rainfall accumulated over 24 consecutive hours |

Rainfall features use a three-hour availability assumption: the latest
accumulation ends three hours before prediction time. Missing rainfall is
not replaced with zero.

Street segment IDs and UTC prediction times identify each row; they are
not numerical model inputs.

### Target Variable (y)

**Target:** `reported_street_or_highway_flooding_next_hour`

Each row represents one physical street segment at one prediction hour.
The target interval is `[prediction time, prediction time + 1 hour)`.

- **1:** At least one qualifying street or highway flooding report is
  uniquely mapped to the segment during the target hour.
- **0:** No qualifying report is observed, with verified source coverage
  and no unresolved evidence preventing a zero label.
- **Unknown:** The label cannot be assigned confidently; exclude it
  from training.

### Training and Evaluation Plan

- Retain the exact chronological boundaries and seven-day gaps defined
  in the experiment configuration.
- Use the configured 2020–2024 period for training, 2025 for validation
  and 2026 for final testing.
- Retain all eligible training positives and sample approximately one
  in 5,000 negatives, recording the corresponding inclusion weights.
- Evaluate at the natural eligible class balance.
- Compare a simple reference predictor, logistic regression and one
  tree-based model.
- Select the model and operating threshold using validation data,
  then evaluate the frozen choice on the test set.

### Hurricane Ida Demonstration

Selected Ida roads and hours will be used to demonstrate model inference
and integration with the cascading engine.

**Ida falls within the training period. Its replay is an integration
demonstration, not an independent evaluation on an unseen storm.**

### Interpretation and Limitations

The model predicts **reported flooding**, which depends on both flooding
conditions and whether someone submits a report. It does not directly
predict water depth, road closure, damage severity or safe passage.

Terrain and rainfall are spatial approximations. Historical rainfall
publication times are not verified, so the availability delay remains
an explicit assumption.

### Notebook Workflow

1. Verify the environment and prepared artifacts.
2. Review coverage, exclusions, labels and rainfall calculations.
3. Freeze the experiment settings and inspect the training sample.
4. Train and compare baseline models.
5. Evaluate the selected model and document its limitations.
6. Export the model and prepare reproducible Ida demonstration inputs
   and predictions.

---

In [2]:
from pathlib import Path
import sys
import json
import pandas as pd

cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in (cwd, *cwd.parents)
        if (path / "config" / "climate_preparation.json").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook inside the Plexa project."
    )

PROCESSED = PROJECT_ROOT / "data" / "processed"

required_files = [
    PROJECT_ROOT / "config" / "climate_preparation.json",
    PROJECT_ROOT / "config" / "climate_baseline_target.json",
    PROCESSED / "baseline_pretraining_report.json",
    PROCESSED / "baseline_dataset_v1" / "dataset.duckdb",
]

missing = [str(path) for path in required_files if not path.is_file()]

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("Project root:", PROJECT_ROOT)

if missing:
    raise FileNotFoundError(
        "Required files are missing:\n" + "\n".join(missing)
    )

print("\nAll required preparation files are present.")
print("Environment check only — no data changed and no training started.")

Python executable: /opt/anaconda3/envs/plexa-climate/bin/python
Python version: 3.11.16
Pandas version: 3.0.6
Project root: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine

All required preparation files are present.
Environment check only — no data changed and no training started.


## 1. Review the Prepared Dataset

Before training, I will inspect the preparation checks, class counts
and configured split boundaries.

These counts describe eligible street-hour combinations, not individual
complaints. Passing automated checks does not replace the remaining
coverage and exclusion review.

In [3]:
from IPython.display import display

with (PROJECT_ROOT / "config" / "climate_preparation.json").open() as file:
    preparation_config = json.load(file)

with (PROJECT_ROOT / "config" / "climate_baseline_target.json").open() as file:
    target_config = json.load(file)

with (PROCESSED / "baseline_pretraining_report.json").open() as file:
    preparation_report = json.load(file)

print("Target:", target_config["target"])
print("Automated checks passed:", preparation_report["checks_passed"])
print("Training performed:", preparation_report["training_performed"])
print(
    "Valid weather cell-hours:",
    f"{preparation_report['valid_weather_fraction']:.2%}",
)

split_table = pd.DataFrame(
    [
        {
            "split": split,
            "start_utc_inclusive": boundaries[0],
            "end_utc_exclusive": boundaries[1],
        }
        for split, boundaries in preparation_config["splits"].items()
    ]
)

print("\nConfigured split boundaries:")
display(split_table)

counts = pd.DataFrame(preparation_report["split_counts"])
counts["split"] = counts["split"].fillna("outside_assigned_splits")

count_table = counts.pivot(
    index="split",
    columns="label",
    values="rows",
).rename(columns={
    0: "no_report_observed",
    1: "report_observed",
})

count_table.columns.name = None
count_table["total_rows"] = (
    count_table["no_report_observed"] + count_table["report_observed"]
)
count_table["positives_per_million"] = (
    count_table["report_observed"] / count_table["total_rows"] * 1_000_000
)

print("\nPrepared population counts — before training sampling:")
display(count_table)

print("\nReviews still required:")
for item in preparation_report["not_complete_until_reviewed"]:
    print("-", item)

Target: reported_street_or_highway_flooding_next_hour
Automated checks passed: True
Training performed: False
Valid weather cell-hours: 98.68%

Configured split boundaries:


,split,start_utc_inclusive,end_utc_exclusive
0,train,2020-10-14T04:00:00Z,2024-12-25T05:00:00Z
1,validation,2025-01-01T05:00:00Z,2025-12-25T05:00:00Z
2,test,2026-01-01T05:00:00Z,2026-10-07T04:00:00Z



Prepared population counts — before training sampling:


,no_report_observed,report_observed,total_rows,positives_per_million
split,,,,
outside_assigned_splits,30692854,78,30692932,2.541302
test,603839002,3768,603842770,6.240035
train,3270165058,12817,3270177875,3.919359
validation,782115939,3383,782119322,4.325427



Reviews still required:
- Assess exclusion bias by borough and storm.
- Review raw archive gaps, source provenance, and static terrain limitations.
- Review Git changes and freeze the dataset.


## 2. Verify Dataset Provenance

I will compare the prepared dataset's recorded configuration and file
hashes with the current files.

This checks whether inputs have changed since preparation. It does not
establish that the data or feature definitions are scientifically correct.
The check is read-only.

In [4]:
import hashlib

dataset_manifest_path = (
    PROCESSED / "baseline_dataset_v1" / "manifest.json"
)

with dataset_manifest_path.open() as file:
    dataset_manifest = json.load(file)

if dataset_manifest["configuration"] != preparation_config:
    raise ValueError(
        "The current configuration differs from the prepared dataset. "
        "Stop here and investigate before training."
    )

recorded_hashes = dataset_manifest.get("input_sha256")

if not recorded_hashes:
    raise ValueError("The dataset manifest contains no input hashes.")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

results = []

for number, (recorded_path, expected_hash) in enumerate(
    recorded_hashes.items(), start=1
):
    path = Path(recorded_path)
    if not path.is_absolute():
        path = PROJECT_ROOT / path

    if not path.is_file():
        status = "MISSING"
    elif sha256_file(path) != expected_hash:
        status = "CHANGED"
    else:
        status = "MATCH"

    results.append({"file": str(path), "status": status})

    if number % 10 == 0 or number == len(recorded_hashes):
        print(f"Checked {number}/{len(recorded_hashes)} files")

integrity_results = pd.DataFrame(results)

print("\nConfiguration: MATCH")
display(
    integrity_results["status"]
    .value_counts()
    .rename_axis("status")
    .to_frame("files")
)

problems = integrity_results[
    integrity_results["status"] != "MATCH"
]

if not problems.empty:
    display(problems)
    raise ValueError(
        "Dataset input verification failed. Investigate before continuing."
    )

print("All recorded dataset input hashes match.")
print("No data changed and no training started.")

Checked 10/227 files
Checked 20/227 files
Checked 30/227 files
Checked 40/227 files
Checked 50/227 files
Checked 60/227 files
Checked 70/227 files
Checked 80/227 files
Checked 90/227 files
Checked 100/227 files
Checked 110/227 files
Checked 120/227 files
Checked 130/227 files
Checked 140/227 files
Checked 150/227 files
Checked 160/227 files
Checked 170/227 files
Checked 180/227 files
Checked 190/227 files
Checked 200/227 files
Checked 210/227 files
Checked 220/227 files
Checked 227/227 files

Configuration: MATCH


,files
status,
MATCH,227


All recorded dataset input hashes match.
No data changed and no training started.


## 3. Monthly Rainfall Feature Coverage

I will review valid weather cell-hours by month and identify months
affected by missing or invalid radar files.

A cell-hour represents one radar grid cell at one prediction hour.
A missing source hour can affect several later accumulation windows.

This check covers rainfall availability only. Street-hour eligibility
also depends on street features, complaint coverage and label uncertainty.

In [5]:
weather_directory = PROCESSED / "baseline_weather_v1"
manifest_paths = sorted(weather_directory.glob("*/manifest.json"))

if not manifest_paths:
    raise FileNotFoundError("No monthly weather manifests found.")

monthly_rows = []

for path in manifest_paths:
    with path.open() as file:
        metadata = json.load(file)

    total = metadata["rows"]
    valid = metadata["valid_rows"]

    monthly_rows.append({
        "month": path.parent.name,
        "total_cell_hours": total,
        "valid_cell_hours": valid,
        "invalid_cell_hours": total - valid,
        "valid_percent": 100 * valid / total if total else float("nan"),
        "absent_source_hours": len(metadata["missing_source_hours"]),
        "invalid_source_files": len(metadata["invalid_source_files"]),
    })

monthly_weather = pd.DataFrame(monthly_rows).sort_values("month")

total_cell_hours = monthly_weather["total_cell_hours"].sum()
valid_cell_hours = monthly_weather["valid_cell_hours"].sum()

print("Monthly partitions:", len(monthly_weather))
print(
    "Overall valid weather cell-hours:",
    f"{valid_cell_hours / total_cell_hours:.4%}",
)

print("\nMonths with incomplete weather features:")
display(
    monthly_weather.loc[
        monthly_weather["invalid_cell_hours"] > 0
    ].reset_index(drop=True).round({"valid_percent": 3})
)

print("\nIda-related monthly coverage:")
display(
    monthly_weather.loc[
        monthly_weather["month"].isin(["2021-08", "2021-09"])
    ].reset_index(drop=True).round({"valid_percent": 3})
)

print(
    "\nSource-hour counts include each partition's lookback period. "
    "Do not sum them to obtain unique missing archive hours."
)

Monthly partitions: 73
Overall valid weather cell-hours: 98.6817%

Months with incomplete weather features:


,month,total_cell_hours,valid_cell_hours,invalid_cell_hours,valid_percent,absent_source_hours,invalid_source_files
0,2020-10,1085836,981819,104017,90.421,18,0
1,2021-03,1887528,1821566,65962,96.505,3,0
2,2021-05,1887528,1824103,63425,96.640,2,0
3,2021-06,1826640,1758141,68499,96.250,4,0
4,2021-07,1887528,1763215,124313,93.414,3,0
5,2021-09,1826640,1760678,65962,96.389,3,0
6,2022-02,1704864,1682031,22833,98.661,4,0
7,2022-03,1887528,1841862,45666,97.581,4,0
8,2022-04,1826640,1643976,182664,90.000,3,0
9,2022-06,1826640,1765752,60888,96.667,1,0



Ida-related monthly coverage:


,month,total_cell_hours,valid_cell_hours,invalid_cell_hours,valid_percent,absent_source_hours,invalid_source_files
0,2021-08,1887528,1887528,0,100.000,0,0
1,2021-09,1826640,1760678,65962,96.389,3,0



Source-hour counts include each partition's lookback period. Do not sum them to obtain unique missing archive hours.


## 4. Rainfall Coverage During the Ida Demonstration Window

The proposed display window is September 1–3, 2021 in America/New_York,
equivalent to [2021-09-01 04:00 UTC, 2021-09-03 04:00 UTC).

I will check the prepared rainfall features for every grid cell and hour
in this window, including duplicate keys, missing values and accumulation
ordering. Street-hour label eligibility will be reviewed separately.

In [6]:
import numpy as np

ida_start = pd.Timestamp("2021-09-01T04:00:00Z")
ida_end = pd.Timestamp("2021-09-03T04:00:00Z")

rain_columns = [
    "rain_1h_mm",
    "rain_3h_mm",
    "rain_6h_mm",
    "rain_24h_mm",
]

weather_path = (
    PROCESSED
    / "baseline_weather_v1"
    / "2021-09"
    / "weather_features.parquet"
)

ida_weather = pd.read_parquet(
    weather_path,
    filters=[
        ("prediction_time_utc", ">=", ida_start.to_pydatetime()),
        ("prediction_time_utc", "<", ida_end.to_pydatetime()),
    ],
)

ida_weather["prediction_time_utc"] = pd.to_datetime(
    ida_weather["prediction_time_utc"], utc=True
)

# Use the reference grid to establish which cells should be present.
reference = pd.read_parquet(
    PROCESSED
    / "radar_download_validation"
    / "radar_20210902_000000.parquet",
    columns=["cell_id", "grid_id"],
)

expected_cells = set(reference["cell_id"])
expected_hours = pd.date_range(
    ida_start, ida_end, freq="h", inclusive="left"
)

duplicate_rows = ida_weather.duplicated(
    ["cell_id", "prediction_time_utc"]
).sum()

values = ida_weather[rain_columns].to_numpy(dtype=float)
finite_rows = np.isfinite(values).all(axis=1)
nonnegative_rows = (values >= 0).all(axis=1)
ordered_rows = (np.diff(values, axis=1) >= -1e-9).all(axis=1)

hourly_counts = (
    ida_weather.groupby("prediction_time_utc")["cell_id"]
    .nunique()
    .reindex(expected_hours, fill_value=0)
)

checks = {
    "expected_rows": len(expected_cells) * len(expected_hours),
    "actual_rows": len(ida_weather),
    "expected_hours": len(expected_hours),
    "observed_hours": ida_weather["prediction_time_utc"].nunique(),
    "duplicate_key_rows": int(duplicate_rows),
    "hours_with_wrong_cell_count": int(
        (hourly_counts != len(expected_cells)).sum()
    ),
    "cell_ids_match_reference": (
        set(ida_weather["cell_id"]) == expected_cells
    ),
    "grid_ids_match_reference": (
        set(ida_weather["grid_id"]) == set(reference["grid_id"])
    ),
    "rows_flagged_invalid": int(
        (~ida_weather["weather_features_valid"].fillna(False)).sum()
    ),
    "rows_with_nonfinite_features": int((~finite_rows).sum()),
    "rows_with_negative_features": int(
        (finite_rows & ~nonnegative_rows).sum()
    ),
    "rows_with_inconsistent_accumulations": int(
        (finite_rows & ~ordered_rows).sum()
    ),
}

display(pd.Series(checks, name="Ida weather checks").to_frame())

,Ida weather checks
expected_rows,121776
actual_rows,121776
expected_hours,48
observed_hours,48
duplicate_key_rows,0
hours_with_wrong_cell_count,0
cell_ids_match_reference,True
grid_ids_match_reference,True
rows_flagged_invalid,0
rows_with_nonfinite_features,0


## 5. Ida Street-Hour Eligibility by Borough

I will count eligible positive and negative street-hours during the
proposed Ida window and compare them with all expected street-hours.

Rows excluded from evaluation may still have usable prediction inputs.
A missing or uncertain label is not evidence that a street was dry.

In [7]:
import duckdb

database_path = (
    PROCESSED / "baseline_dataset_v1" / "dataset.duckdb"
)

connection = duckdb.connect(str(database_path), read_only=True)

try:
    connection.execute("SET threads=2")
    connection.execute("SET memory_limit='2GB'")
    connection.execute("SET TimeZone='UTC'")

    ida_borough_review = connection.execute(
        """
        WITH street_totals AS (
            SELECT
                coalesce(borough_at_midpoint, 'Unassigned') AS borough,
                count(*) AS physical_segments
            FROM street_features
            GROUP BY 1
        ),
        eligible_counts AS (
            SELECT
                coalesce(s.borough_at_midpoint, 'Unassigned') AS borough,
                count(*) FILTER (WHERE e.label = 1)
                    AS positive_street_hours,
                count(*) FILTER (WHERE e.label = 0)
                    AS negative_street_hours
            FROM eligible e
            JOIN street_features s USING (street_segment_id)
            WHERE e.prediction_time_utc >= ?::TIMESTAMPTZ
              AND e.prediction_time_utc < ?::TIMESTAMPTZ
            GROUP BY 1
        )
        SELECT
            s.borough,
            s.physical_segments,
            coalesce(e.positive_street_hours, 0)
                AS positive_street_hours,
            coalesce(e.negative_street_hours, 0)
                AS negative_street_hours
        FROM street_totals s
        LEFT JOIN eligible_counts e USING (borough)
        ORDER BY s.borough
        """,
        [ida_start.isoformat(), ida_end.isoformat()],
    ).fetchdf()

finally:
    connection.close()

hours_in_window = len(expected_hours)

ida_borough_review["expected_street_hours"] = (
    ida_borough_review["physical_segments"] * hours_in_window
)

ida_borough_review["eligible_street_hours"] = (
    ida_borough_review["positive_street_hours"]
    + ida_borough_review["negative_street_hours"]
)

ida_borough_review["excluded_street_hours"] = (
    ida_borough_review["expected_street_hours"]
    - ida_borough_review["eligible_street_hours"]
)

ida_borough_review["eligible_percent"] = (
    100
    * ida_borough_review["eligible_street_hours"]
    / ida_borough_review["expected_street_hours"]
)

display(ida_borough_review.round({"eligible_percent": 2}))

assert (
    ida_borough_review["excluded_street_hours"] >= 0
).all(), "Unexpected counts: investigate duplicate joins."

print(
    "These are street-hour counts, not unique complaint or storm counts."
)
print(
    "Excluded rows combine feature, coverage and label-uncertainty "
    "conditions. We will separate the reasons next."
)

,borough,physical_segments,positive_street_hours,negative_street_hours,expected_street_hours,eligible_street_hours,excluded_street_hours,eligible_percent
0,Bronx,12769,54,395830,612912,395884,217028,64.59
1,Brooklyn,21601,160,669587,1036848,669747,367101,64.59
2,Manhattan,8247,17,255654,395856,255671,140185,64.59
3,Queens,35843,239,1111061,1720464,1111300,609164,64.59
4,Staten Island,13360,166,414125,641280,414291,226989,64.60
5,outside_borough_polygons,75,0,2325,3600,2325,1275,64.58


These are street-hour counts, not unique complaint or storm counts.
Excluded rows combine feature, coverage and label-uncertainty conditions. We will separate the reasons next.


## 6. Explain Ida Label Exclusions

I will inspect source coverage and uncertainty masks for each Ida hour.

A global uncertainty mask blocks negative labels across all segments
for that hour. A definite positive can still be retained. These masks
affect label eligibility, not necessarily the ability to run inference.

In [8]:
connection = duckdb.connect(str(database_path), read_only=True)

try:
    connection.execute("SET TimeZone='UTC'")

    ida_hour_review = connection.execute(
        """
        WITH hours AS (
            SELECT prediction_time_utc
            FROM generate_series(
                ?::TIMESTAMPTZ,
                ?::TIMESTAMPTZ - INTERVAL '1 hour',
                INTERVAL '1 hour'
            ) AS h(prediction_time_utc)
        ),
        positive_counts AS (
            SELECT
                prediction_time_utc,
                count(*) AS positive_segment_hours
            FROM positive
            WHERE prediction_time_utc >= ?::TIMESTAMPTZ
              AND prediction_time_utc < ?::TIMESTAMPTZ
            GROUP BY 1
        ),
        segment_mask_counts AS (
            SELECT
                prediction_time_utc,
                count(*) AS segment_unknown_pairs
            FROM segment_unknown
            WHERE prediction_time_utc >= ?::TIMESTAMPTZ
              AND prediction_time_utc < ?::TIMESTAMPTZ
            GROUP BY 1
        )
        SELECT
            h.prediction_time_utc::VARCHAR AS prediction_time_utc,
            c.prediction_time_utc IS NOT NULL
                AS source_coverage_verified,
            g.prediction_time_utc IS NOT NULL
                AS global_unknown_mask,
            coalesce(p.positive_segment_hours, 0)
                AS positive_segment_hours,
            coalesce(u.segment_unknown_pairs, 0)
                AS segment_unknown_pairs
        FROM hours h
        LEFT JOIN coverage c USING (prediction_time_utc)
        LEFT JOIN global_unknown g USING (prediction_time_utc)
        LEFT JOIN positive_counts p USING (prediction_time_utc)
        LEFT JOIN segment_mask_counts u USING (prediction_time_utc)
        ORDER BY h.prediction_time_utc
        """,
        [ida_start.isoformat(), ida_end.isoformat()] * 3,
    ).fetchdf()

finally:
    connection.close()

print("Hours reviewed:", len(ida_hour_review))
print(
    "Hours without verified source coverage:",
    int((~ida_hour_review["source_coverage_verified"]).sum()),
)
print(
    "Hours with global uncertainty masks:",
    int(ida_hour_review["global_unknown_mask"].sum()),
)
print(
    "Segment-specific uncertainty pairs:",
    int(ida_hour_review["segment_unknown_pairs"].sum()),
)

affected_hours = ida_hour_review.loc[
    ~ida_hour_review["source_coverage_verified"]
    | ida_hour_review["global_unknown_mask"]
    | (ida_hour_review["segment_unknown_pairs"] > 0)
]

display(affected_hours.reset_index(drop=True))

Hours reviewed: 48
Hours without verified source coverage: 0
Hours with global uncertainty masks: 17
Segment-specific uncertainty pairs: 4


,prediction_time_utc,source_coverage_verified,global_unknown_mask,positive_segment_hours,segment_unknown_pairs
0,2021-09-01 23:00:00+00,True,True,6,2
1,2021-09-02 00:00:00+00,True,False,13,2
2,2021-09-02 01:00:00+00,True,True,57,0
3,2021-09-02 02:00:00+00,True,True,145,0
4,2021-09-02 03:00:00+00,True,True,68,0
5,2021-09-02 05:00:00+00,True,True,43,0
6,2021-09-02 06:00:00+00,True,True,14,0
7,2021-09-02 07:00:00+00,True,True,4,0
8,2021-09-02 10:00:00+00,True,True,5,0
9,2021-09-02 12:00:00+00,True,True,12,0


## 7. Trace the Reports Behind Ida Uncertainty Masks

I will inspect qualifying reports that could not be assigned uniquely
to a physical street segment.

This identifies the source of location uncertainty without changing
the labels or relaxing the existing rules.

In [9]:
audit_path = (
    PROCESSED
    / "physical_311_mapping_v1"
    / "report_audit.parquet"
)

ida_report_audit = pd.read_parquet(
    audit_path,
    filters=[
        ("created_at", ">=", ida_start.to_pydatetime()),
        ("created_at", "<", ida_end.to_pydatetime()),
    ],
)

ida_report_audit["created_at"] = pd.to_datetime(
    ida_report_audit["created_at"], utc=True
)

ida_report_audit["report_hour_utc"] = (
    ida_report_audit["created_at"].dt.floor("h")
)

print("Qualifying reports by mapping status:")
display(
    ida_report_audit.groupby("match_status")
    .agg(
        unique_reports=("unique_key", "nunique"),
        affected_hours=("report_hour_utc", "nunique"),
    )
    .reset_index()
)

global_mask_reports = ida_report_audit.loc[
    ida_report_audit["match_status"].isin([
        "missing_or_invalid_coordinates",
        "outside_match_distance",
    ])
]

print("\nReports that trigger global location-uncertainty masks:")
display(
    global_mask_reports.groupby(
        ["report_hour_utc", "match_status"]
    )
    .agg(unique_reports=("unique_key", "nunique"))
    .reset_index()
)

print(
    "\nDistinct hours explained by these location problems:",
    global_mask_reports["report_hour_utc"].nunique(),
)

print("\nDescriptor breakdown:")
display(
    global_mask_reports.groupby(
        ["match_status", "descriptor"]
    )
    .agg(unique_reports=("unique_key", "nunique"))
    .reset_index()
)

Qualifying reports by mapping status:


,match_status,unique_reports,affected_hours
0,ambiguous_segment,2,2
1,matched_unique_segment,685,35
2,missing_or_invalid_coordinates,23,17
3,outside_match_distance,5,3



Reports that trigger global location-uncertainty masks:


,report_hour_utc,match_status,unique_reports
0,2021-09-01 23:00:00+00:00,missing_or_invalid_coordinates,1
1,2021-09-02 01:00:00+00:00,missing_or_invalid_coordinates,3
2,2021-09-02 02:00:00+00:00,missing_or_invalid_coordinates,2
3,2021-09-02 02:00:00+00:00,outside_match_distance,2
4,2021-09-02 03:00:00+00:00,missing_or_invalid_coordinates,2
5,2021-09-02 03:00:00+00:00,outside_match_distance,2
6,2021-09-02 05:00:00+00:00,missing_or_invalid_coordinates,2
7,2021-09-02 06:00:00+00:00,missing_or_invalid_coordinates,1
8,2021-09-02 07:00:00+00:00,missing_or_invalid_coordinates,1
9,2021-09-02 10:00:00+00:00,missing_or_invalid_coordinates,1



Distinct hours explained by these location problems: 17

Descriptor breakdown:


,match_status,descriptor,unique_reports
0,missing_or_invalid_coordinates,Highway Flooding (SH),21
1,missing_or_invalid_coordinates,Street Flooding (SJ),2
2,outside_match_distance,Street Flooding (SJ),5


### Ida Location-Uncertainty Review

Within the proposed 48-hour Ida display window:

- 685 qualifying reports were uniquely matched to physical segments.
- 2 reports had ambiguous segment matches.
- 23 reports lacked valid coordinates.
- 5 reports were outside the configured matching distance.

The 28 unlocated/out-of-distance reports span 17 distinct hours,
matching the number of global uncertainty-mask hours.

Of the 23 reports without valid coordinates, 21 concern highway flooding
and 2 concern street flooding. All 5 out-of-distance reports concern
street flooding.

Under the current policy, global masks prevent negative labels during
these hours, while definite positives remain eligible. This explains
most of the reduction to approximately 64.6% eligible Ida street-hours.

This is a conservative labeling policy with a potential selection bias.
Inverse sampling weights do not correct exclusions caused by uncertain
labels. Results must be described as performance on the eligible
population, not all NYC street-hours.

No label rules were changed during this review. Valid feature rows with
unknown labels may still receive demonstration predictions, but cannot
be treated as verified negatives when evaluating the model.

## 8. Inspect Original Radar Time Metadata

Before tracing the rainfall windows, I will inspect the original Ida
GRIB metadata.

Matching a filename to the GRIB valid time confirms timestamp consistency.
It does not, by itself, establish the rainfall accumulation interval.

If the GRIB does not describe the interval, I must verify the product's
time convention using authoritative source documentation.

In [10]:
import eccodes

grib_files = sorted(
    (PROJECT_ROOT / "data" / "raw" / "radar_ida").glob("*.grib2")
)

if not grib_files:
    raise FileNotFoundError("Original Ida GRIB files were not found.")

metadata_keys = [
    "stepType",
    "stepUnits",
    "startStep",
    "endStep",
    "stepRange",
    "lengthOfTimeRange",
    "indicatorOfUnitForTimeRange",
    "dataDate",
    "dataTime",
    "validityDate",
    "validityTime",
    "units",
]

metadata_rows = []

for path in grib_files:
    with path.open("rb") as file:
        message = eccodes.codes_grib_new_from_file(file)

        if message is None:
            raise ValueError(f"No GRIB message found: {path.name}")

        try:
            row = {"source_file": path.name}

            for key in metadata_keys:
                try:
                    row[key] = eccodes.codes_get(message, key)
                except eccodes.CodesInternalError:
                    row[key] = None

            metadata_rows.append(row)

        finally:
            eccodes.codes_release(message)

radar_time_metadata = pd.DataFrame(metadata_rows)

display(radar_time_metadata)

print(
    "These values describe the available GRIB metadata. "
    "Accumulation-end semantics still need explicit evidence."
)

,source_file,stepType,stepUnits,startStep,endStep,stepRange,lengthOfTimeRange,indicatorOfUnitForTimeRange,dataDate,dataTime,validityDate,validityTime,units
0,MultiSensor_QPE_01H_Pass2_00.00_20210902-00000...,instant,0,0m,0m,0m,None,None,20210902,0,20210902,0,unknown
1,MultiSensor_QPE_01H_Pass2_00.00_20210902-01000...,instant,0,0m,0m,0m,None,None,20210902,100,20210902,100,unknown
2,MultiSensor_QPE_01H_Pass2_00.00_20210902-02000...,instant,0,0m,0m,0m,None,None,20210902,200,20210902,200,unknown
3,MultiSensor_QPE_01H_Pass2_00.00_20210902-03000...,instant,0,0m,0m,0m,None,None,20210902,300,20210902,300,unknown
4,MultiSensor_QPE_01H_Pass2_00.00_20210902-04000...,instant,0,0m,0m,0m,None,None,20210902,400,20210902,400,unknown
5,MultiSensor_QPE_01H_Pass2_00.00_20210902-05000...,instant,0,0m,0m,0m,None,None,20210902,500,20210902,500,unknown


These values describe the available GRIB metadata. Accumulation-end semantics still need explicit evidence.


## 9. Trace One Ida Rainfall Calculation

The NOAA Multi-Sensor QPE guide states that accumulations end at the
indicated time. The operational product table identifies the 01H
accumulation unit as millimetres.

The local GRIB metadata does not independently encode the accumulation
duration in the inspected fields. Its interpretation therefore relies
on the product documentation.

Sources:
- https://vlab.noaa.gov/web/wdtd/-/multi-sensor-qpe
- https://www.nssl.noaa.gov/projects/mrms/operational/tables.php

For a prediction at 2021-09-02 03:00 UTC, the configured three-hour delay
allows rainfall accumulations ending no later than 00:00 UTC.

I will select one street deterministically, list the 24 hourly source
intervals for its radar cell, and compare their sums with the prepared
1-, 3-, 6- and 24-hour features.

This checks one worked example. It does not verify historical publication
times or establish provenance for every archive version.

In [11]:
worked_prediction_time = pd.Timestamp("2021-09-02T03:00:00Z")
lag_hours = preparation_config["rainfall_availability_lag_hours"]
cutoff = worked_prediction_time - pd.Timedelta(hours=lag_hours)

# Choose a valid street deterministically, without looking at its label.
street_features = pd.read_parquet(
    PROCESSED
    / "baseline_street_features_v1"
    / "street_features.parquet"
)

worked_street = (
    street_features.loc[street_features["static_features_valid"]]
    .sort_values("street_segment_id")
    .iloc[0]
)

cell_id = int(worked_street["cell_id"])
grid_id = worked_street["grid_id"]

source_end_times = pd.date_range(
    end=cutoff,
    periods=24,
    freq="h",
)

source_rows = []

for interval_end in source_end_times:
    filename = f"radar_{interval_end:%Y%m%d_%H%M%S}.parquet"
    path = PROJECT_ROOT / "data" / "raw" / "radar_parquet_v2" / filename

    frame = pd.read_parquet(path)
    selected = frame.loc[
        (frame["cell_id"] == cell_id)
        & (frame["grid_id"] == grid_id)
    ]

    if len(selected) != 1:
        raise ValueError(f"Expected one matching cell in {filename}")

    row = selected.iloc[0]
    actual_time = pd.to_datetime(row["ts_utc"], utc=True)
    rainfall = float(row["rain_mm"])

    if actual_time != interval_end:
        raise ValueError(f"Timestamp mismatch in {filename}")

    if (
        pd.isna(row["is_missing"])
        or bool(row["is_missing"])
        or not np.isfinite(rainfall)
        or rainfall < 0
    ):
        raise ValueError(f"Missing or invalid rainfall in {filename}")

    source_rows.append({
        "interval_start_utc": interval_end - pd.Timedelta(hours=1),
        "interval_end_utc": interval_end,
        "rain_mm": rainfall,
        "source_file": filename,
    })

worked_rainfall = pd.DataFrame(source_rows)

prepared_rows = ida_weather.loc[
    (ida_weather["prediction_time_utc"] == worked_prediction_time)
    & (ida_weather["cell_id"] == cell_id)
    & (ida_weather["grid_id"] == grid_id)
]

if len(prepared_rows) != 1:
    raise ValueError("Expected one matching prepared weather row.")

prepared_row = prepared_rows.iloc[0]
comparison_rows = []

for hours in preparation_config["rainfall_windows_hours"]:
    feature = f"rain_{hours}h_mm"
    contributing = worked_rainfall.tail(hours)
    calculated = float(contributing["rain_mm"].sum())
    prepared = float(prepared_row[feature])

    comparison_rows.append({
        "feature": feature,
        "first_interval_start_utc": contributing.iloc[0]["interval_start_utc"],
        "last_interval_end_utc": contributing.iloc[-1]["interval_end_utc"],
        "calculated_mm": calculated,
        "prepared_mm": prepared,
        "matches": bool(np.isclose(
            calculated, prepared, rtol=1e-7, atol=1e-5
        )),
    })

rainfall_comparison = pd.DataFrame(comparison_rows)

print("Street:", worked_street["street_segment_id"])
print("Radar cell:", cell_id)
print("Prediction time:", worked_prediction_time)
print("Latest permitted rainfall end:", cutoff)

display(worked_rainfall)
display(rainfall_comparison)

assert rainfall_comparison["matches"].all(), (
    "A rainfall calculation differs from the prepared feature."
)

print("All four worked rainfall sums match the prepared features.")

Street: 0001b625d0510c53ae2934b2ef3e48d5da7f636589dd088ad0d20ff47f854b9e
Radar cell: 1324
Prediction time: 2021-09-02 03:00:00+00:00
Latest permitted rainfall end: 2021-09-02 00:00:00+00:00


,interval_start_utc,interval_end_utc,rain_mm,source_file
0,2021-09-01 00:00:00+00:00,2021-09-01 01:00:00+00:00,0.0,radar_20210901_010000.parquet
1,2021-09-01 01:00:00+00:00,2021-09-01 02:00:00+00:00,0.0,radar_20210901_020000.parquet
2,2021-09-01 02:00:00+00:00,2021-09-01 03:00:00+00:00,0.0,radar_20210901_030000.parquet
3,2021-09-01 03:00:00+00:00,2021-09-01 04:00:00+00:00,0.0,radar_20210901_040000.parquet
4,2021-09-01 04:00:00+00:00,2021-09-01 05:00:00+00:00,0.0,radar_20210901_050000.parquet
5,2021-09-01 05:00:00+00:00,2021-09-01 06:00:00+00:00,0.0,radar_20210901_060000.parquet
6,2021-09-01 06:00:00+00:00,2021-09-01 07:00:00+00:00,0.0,radar_20210901_070000.parquet
7,2021-09-01 07:00:00+00:00,2021-09-01 08:00:00+00:00,1.0,radar_20210901_080000.parquet
8,2021-09-01 08:00:00+00:00,2021-09-01 09:00:00+00:00,0.3,radar_20210901_090000.parquet
9,2021-09-01 09:00:00+00:00,2021-09-01 10:00:00+00:00,0.0,radar_20210901_100000.parquet


,feature,first_interval_start_utc,last_interval_end_utc,calculated_mm,prepared_mm,matches
0,rain_1h_mm,2021-09-01 23:00:00+00:00,2021-09-02 00:00:00+00:00,11.600000,11.600000,True
1,rain_3h_mm,2021-09-01 21:00:00+00:00,2021-09-02 00:00:00+00:00,23.700001,23.700001,True
2,rain_6h_mm,2021-09-01 18:00:00+00:00,2021-09-02 00:00:00+00:00,23.700001,23.700001,True
3,rain_24h_mm,2021-09-01 00:00:00+00:00,2021-09-02 00:00:00+00:00,27.700001,27.700001,True


All four worked rainfall sums match the prepared features.


## 10. Retention of Positive Labels by Month and Borough

I will check which prepared positive segment-hours remain eligible after
street-feature, weather-feature and source-coverage checks.

This identifies whether missing data removes positive examples unevenly.
Positive segment-hours are not counts of independent storms.

Exclusion reasons below are assigned in a fixed order, so each positive
row appears in exactly one category.

In [12]:
connection = duckdb.connect(str(database_path), read_only=True)

try:
    connection.execute("SET threads=2")
    connection.execute("SET memory_limit='2GB'")
    connection.execute("SET TimeZone='UTC'")

    positive_retention = connection.execute(
        """
        WITH reviewed AS (
            SELECT
                strftime(
                    p.prediction_time_utc, '%Y-%m'
                ) AS month,
                coalesce(
                    s.borough_at_midpoint, 'Unassigned'
                ) AS borough,
                CASE
                    WHEN s.street_segment_id IS NULL
                        THEN 'missing_street'
                    WHEN s.static_features_valid IS NOT TRUE
                        THEN 'invalid_static_features'
                    WHEN w.prediction_time_utc IS NULL
                        THEN 'missing_weather_row'
                    WHEN w.weather_features_valid IS NOT TRUE
                        THEN 'invalid_weather_features'
                    WHEN c.prediction_time_utc IS NULL
                        THEN 'unverified_source_coverage'
                    ELSE 'eligible_positive'
                END AS review_status
            FROM positive p
            LEFT JOIN street_features s
                USING (street_segment_id)
            LEFT JOIN weather w
                ON s.cell_id = w.cell_id
               AND s.grid_id = w.grid_id
               AND p.prediction_time_utc = w.prediction_time_utc
            LEFT JOIN coverage c
                ON p.prediction_time_utc = c.prediction_time_utc
        )
        SELECT
            month,
            borough,
            review_status,
            count(*) AS positive_segment_hours
        FROM reviewed
        GROUP BY month, borough, review_status
        ORDER BY month, borough, review_status
        """
    ).fetchdf()

    expected_positive_count = connection.execute(
        "SELECT count(*) FROM positive"
    ).fetchone()[0]

finally:
    connection.close()

assert (
    positive_retention["positive_segment_hours"].sum()
    == expected_positive_count
), "Counts do not reconcile; investigate the joins."

print("All prepared positive segment-hours, including split gaps:")
display(
    positive_retention.groupby("review_status")[
        "positive_segment_hours"
    ].sum().to_frame()
)

print("\nPositive exclusions by month and borough:")
positive_exclusions = positive_retention.loc[
    positive_retention["review_status"] != "eligible_positive"
]
display(positive_exclusions.reset_index(drop=True))

print("\nSeptember 2022 positive-label retention:")
display(
    positive_retention.loc[
        positive_retention["month"] == "2022-09"
    ].reset_index(drop=True)
)

All prepared positive segment-hours, including split gaps:


,positive_segment_hours
review_status,
eligible_positive,20046
invalid_weather_features,316



Positive exclusions by month and borough:


,month,borough,review_status,positive_segment_hours
0,2020-10,Bronx,invalid_weather_features,3
1,2020-10,Brooklyn,invalid_weather_features,6
2,2020-10,Manhattan,invalid_weather_features,3
3,2020-10,Queens,invalid_weather_features,16
4,2020-10,Staten Island,invalid_weather_features,6
5,2021-03,Bronx,invalid_weather_features,1
6,2021-03,Brooklyn,invalid_weather_features,1
7,2021-03,Manhattan,invalid_weather_features,1
8,2021-03,Queens,invalid_weather_features,3
9,2021-06,Brooklyn,invalid_weather_features,4



September 2022 positive-label retention:


,month,borough,review_status,positive_segment_hours
0,2022-09,Bronx,eligible_positive,14
1,2022-09,Bronx,invalid_weather_features,5
2,2022-09,Brooklyn,eligible_positive,75
3,2022-09,Brooklyn,invalid_weather_features,18
4,2022-09,Manhattan,eligible_positive,14
5,2022-09,Manhattan,invalid_weather_features,4
6,2022-09,Queens,eligible_positive,106
7,2022-09,Queens,invalid_weather_features,28
8,2022-09,Staten Island,eligible_positive,25
9,2022-09,Staten Island,invalid_weather_features,6


## 11. Verify the Actual Training Sample

The prepared population retains 20,046 of 20,362 positive segment-hours.
Weather-feature gaps exclude 316 positives, including 61 in September 2022.
These counts include all configured periods and split gaps.

I will now inspect the deterministic training sample. All eligible training
positives should be retained. Negatives are sampled at approximately
1 in 5,000, with inclusion weights of 5,000; positive weights are 1.

These weights address deliberate negative sampling. They do not correct
the selection effects caused by uncertain labels or missing features.

In [13]:
connection = duckdb.connect(str(database_path), read_only=True)

try:
    connection.execute("SET threads=2")
    connection.execute("SET memory_limit='2GB'")

    training_sample_summary = connection.execute(
        """
        SELECT
            label,
            count(*) AS sampled_rows,
            min(sample_weight) AS minimum_weight,
            max(sample_weight) AS maximum_weight,
            sum(sample_weight) AS represented_population_estimate
        FROM training_sample
        GROUP BY label
        ORDER BY label
        """
    ).fetchdf()

finally:
    connection.close()

display(training_sample_summary)

actual_sample_size = int(
    training_sample_summary["sampled_rows"].sum()
)

positive_sample_size = int(
    training_sample_summary.loc[
        training_sample_summary["label"] == 1,
        "sampled_rows",
    ].sum()
)

print(f"Actual training sample: {actual_sample_size:,} rows")
print(f"Positive training rows: {positive_sample_size:,}")
print(
    "Sample positive percentage:",
    f"{100 * positive_sample_size / actual_sample_size:.3f}%",
)

expected_training_positives = int(
    counts.loc[
        (counts["split"] == "train") & (counts["label"] == 1),
        "rows",
    ].sum()
)

assert positive_sample_size == expected_training_positives, (
    "The sample does not retain all recorded training positives."
)

expected_weights = {
    0: round(1 / preparation_config["training_negative_keep_fraction"]),
    1: 1,
}

for row in training_sample_summary.itertuples(index=False):
    expected = expected_weights[int(row.label)]
    assert row.minimum_weight == row.maximum_weight == expected, (
        f"Unexpected weights for label {row.label}"
    )

print("Positive retention and sampling-weight checks passed.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,label,sampled_rows,minimum_weight,maximum_weight,represented_population_estimate
0,0,654040,5000.0,5000.0,3.270200e+09
1,1,12817,1.0,1.0,1.281700e+04


Actual training sample: 666,857 rows
Positive training rows: 12,817
Sample positive percentage: 1.922%
Positive retention and sampling-weight checks passed.


### Training Sample Review

The deterministic training sample contains 666,857 street-hour rows:

- 654,040 negatives, each with inclusion weight 5,000.
- 12,817 positives, each with inclusion weight 1.
- Positive share of the sample: approximately 1.922%.

All eligible training positives are retained. The sample balance differs
from the natural population, so fitting must account for the sampling
policy. Sampling weights alone do not establish calibrated probabilities.

Validation and test evaluation will preserve natural eligible prevalence.

## 12. Eligible Street-Hours by Month and Borough

I will compare eligible positive and negative rows with the expected
street-hour population for each UTC month and borough.

Excluded rows may reflect missing features, unverified source coverage
or uncertain labels. These counts do not establish complete historical
street coverage.

This review includes the configured snapshot, including split gaps.

In [14]:
connection = duckdb.connect(str(database_path), read_only=True)

try:
    connection.execute("SET threads=2")
    connection.execute("SET memory_limit='2GB'")
    connection.execute("SET TimeZone='UTC'")

    monthly_borough_coverage = connection.execute(
        """
        WITH hours_by_month AS (
            SELECT
                strftime(hour_utc, '%Y-%m') AS month,
                count(*) AS hours_in_snapshot
            FROM generate_series(
                ?::TIMESTAMPTZ,
                ?::TIMESTAMPTZ - INTERVAL '1 hour',
                INTERVAL '1 hour'
            ) AS h(hour_utc)
            GROUP BY 1
        ),
        streets_by_borough AS (
            SELECT
                coalesce(borough_at_midpoint, 'Unassigned') AS borough,
                count(*) AS physical_segments
            FROM street_features
            GROUP BY 1
        ),
        eligible_counts AS (
            SELECT
                strftime(e.prediction_time_utc, '%Y-%m') AS month,
                coalesce(s.borough_at_midpoint, 'Unassigned') AS borough,
                count(*) FILTER (WHERE e.label = 1) AS positive_rows,
                count(*) FILTER (WHERE e.label = 0) AS negative_rows
            FROM eligible e
            JOIN street_features s USING (street_segment_id)
            GROUP BY 1, 2
        )
        SELECT
            h.month,
            s.borough,
            s.physical_segments * h.hours_in_snapshot
                AS expected_street_hours,
            coalesce(e.positive_rows, 0) AS positive_rows,
            coalesce(e.negative_rows, 0) AS negative_rows
        FROM hours_by_month h
        CROSS JOIN streets_by_borough s
        LEFT JOIN eligible_counts e
            ON h.month = e.month AND s.borough = e.borough
        ORDER BY h.month, s.borough
        """,
        [
            preparation_config["snapshot_start_utc"],
            preparation_config["snapshot_end_utc_exclusive"],
        ],
    ).fetchdf()

finally:
    connection.close()

monthly_borough_coverage["eligible_rows"] = (
    monthly_borough_coverage["positive_rows"]
    + monthly_borough_coverage["negative_rows"]
)

monthly_borough_coverage["excluded_rows"] = (
    monthly_borough_coverage["expected_street_hours"]
    - monthly_borough_coverage["eligible_rows"]
)

monthly_borough_coverage["eligible_percent"] = (
    100
    * monthly_borough_coverage["eligible_rows"]
    / monthly_borough_coverage["expected_street_hours"]
)

assert (monthly_borough_coverage["excluded_rows"] >= 0).all()

assert (
    monthly_borough_coverage["eligible_rows"].sum()
    == counts["rows"].sum()
), "The coverage totals differ from the saved preparation report."

print("Twenty month/borough combinations with lowest eligibility:")
display(
    monthly_borough_coverage.nsmallest(20, "eligible_percent")
    .reset_index(drop=True)
    .round({"eligible_percent": 2})
)

print("\nSeptember 2022:")
display(
    monthly_borough_coverage.loc[
        monthly_borough_coverage["month"] == "2022-09"
    ].reset_index(drop=True).round({"eligible_percent": 2})
)

print("\nCoverage totals reconcile with the preparation report.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Twenty month/borough combinations with lowest eligibility:


,month,borough,expected_street_hours,positive_rows,negative_rows,eligible_rows,excluded_rows,eligible_percent
0,2022-09,Manhattan,5937840,14,4643047,4643061,1294779,78.19
1,2022-09,outside_borough_polygons,54000,0,42225,42225,11775,78.19
2,2022-09,Staten Island,9619200,25,7521658,7521683,2097517,78.19
3,2022-09,Bronx,9193680,14,7188936,7188950,2004730,78.19
4,2022-09,Queens,25806960,106,20179526,20179632,5627328,78.19
5,2022-09,Brooklyn,15552720,75,12161306,12161381,3391339,78.19
6,2022-04,Bronx,9193680,14,8133839,8133853,1059827,88.47
7,2022-04,Manhattan,5937840,15,5253324,5253339,684501,88.47
8,2022-04,outside_borough_polygons,54000,0,47775,47775,6225,88.47
9,2022-04,Queens,25806960,77,22831919,22831996,2974964,88.47



September 2022:


,month,borough,expected_street_hours,positive_rows,negative_rows,eligible_rows,excluded_rows,eligible_percent
0,2022-09,Bronx,9193680,14,7188936,7188950,2004730,78.19
1,2022-09,Brooklyn,15552720,75,12161306,12161381,3391339,78.19
2,2022-09,Manhattan,5937840,14,4643047,4643061,1294779,78.19
3,2022-09,Queens,25806960,106,20179526,20179632,5627328,78.19
4,2022-09,Staten Island,9619200,25,7521658,7521683,2097517,78.19
5,2022-09,outside_borough_polygons,54000,0,42225,42225,11775,78.19



Coverage totals reconcile with the preparation report.


## 13. Data Review Conclusions

The preparation checks and recorded input hashes passed.

- The training sample contains 666,857 rows: 12,817 positives and
  654,040 sampled negatives.
- Overall weather-feature validity is 98.6817%.
- Missing weather features exclude 316 of 20,362 prepared positive
  segment-hours.
- September 2022 has the lowest monthly street-hour eligibility,
  approximately 78.19%, and loses 61 positive segment-hours.
- The proposed Ida window has complete prepared rainfall features.
  Label uncertainty reduces its eligible street-hours to about 64.6%.
- Ida location problems explain all 17 global uncertainty-mask hours.
- One worked rainfall calculation matches all four prepared windows.

These checks support proceeding with a baseline experiment, subject to
the recorded limitations. Results describe the eligible population;
they do not establish performance across every NYC street-hour.

Inverse sampling weights address deliberate negative sampling, but do
not correct exclusions caused by missing features or uncertain labels.

Ida remains an integration replay because it lies within training.
No data, labels or existing split boundaries were changed during review.

In [15]:
experiment_path = (
    PROJECT_ROOT / "config" / "climate_baseline_experiment_v2.json"
)

experiment = {
    "experiment_id": "climate-baseline-v2-chronological-01",
    "target": target_config["target"],
    "features": preparation_config["features"],
    "splits": preparation_config["splits"],
    "rainfall_availability_lag_hours": (
        preparation_config["rainfall_availability_lag_hours"]
    ),
    "training_sample": {
        "view": "training_sample",
        "actual_rows": 666857,
        "positive_rows": 12817,
        "negative_rows": 654040,
        "negative_keep_fraction": (
            preparation_config["training_negative_keep_fraction"]
        ),
        "sampling_seed": preparation_config["training_sampling_seed"],
        "positive_inclusion_weight": 1,
        "negative_inclusion_weight": 5000,
        "additional_balanced_class_weights": False,
    },
    "random_seed": 42,
    "candidate_families": [
        "training_prior_reference",
        "logistic_regression",
        "one_tree_based_model",
    ],
    "selection_policy": {
        "model_selection_data": "validation_only",
        "threshold_selection_data": "validation_only",
        "final_test": "evaluate_frozen_model_and_threshold",
        "evaluation_population": "natural_eligible_prevalence",
        "metrics": [
            "average_precision",
            "precision_recall",
            "precision_and_recall_at_selected_threshold",
            "true_positive_false_positive_false_negative_counts",
            "alerts_per_hour_or_day",
        ],
        "batch_average_precision_averaging_allowed": False,
        "probability_calibration_claimed": False,
    },
    "ida": {
        "mode": "integration_replay",
        "included_in_training_period": True,
        "independent_event_evaluation": False,
        "proposed_start_utc": ida_start.isoformat(),
        "proposed_end_utc_exclusive": ida_end.isoformat(),
        "shared_road_subset_frozen": False,
    },
    "preparation_evidence_sha256": {
        "dataset_manifest": sha256_file(dataset_manifest_path),
        "preparation_config": sha256_file(
            PROJECT_ROOT / "config" / "climate_preparation.json"
        ),
        "target_config": sha256_file(
            PROJECT_ROOT / "config" / "climate_baseline_target.json"
        ),
        "pretraining_report": sha256_file(
            PROCESSED / "baseline_pretraining_report.json"
        ),
    },
    "status": "review_recorded_model_settings_pending",
}

# Preserve any existing version rather than silently replacing it.
if experiment_path.exists():
    with experiment_path.open() as file:
        existing_experiment = json.load(file)

    if existing_experiment != experiment:
        raise FileExistsError(
            "An experiment file already exists with different settings. "
            "Review the difference before changing it."
        )

    print("Existing experiment plan matches.")
else:
    with experiment_path.open("x") as file:
        json.dump(experiment, file, indent=2)
        file.write("\n")

    print("Experiment plan saved:", experiment_path)

print("Next: specify model settings and preprocessing before fitting.")

Experiment plan saved: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/config/climate_baseline_experiment_v2.json
Next: specify model settings and preprocessing before fitting.


## 14. Baseline Model Settings

I will compare a weighted training-prior reference, logistic regression
and histogram gradient boosting using the same features and split.

Logistic regression will use StandardScaler fitted on training rows
with their sampling weights. The tree model will use the original
feature values.

Inclusion weights will be divided by their training-sample mean before
fitting. This preserves relative weights while defining their scale
for regularization. No additional balanced class weights will be used.

Gradient boosting will use a fixed iteration count with automatic
early stopping disabled. Model comparison and operating-threshold
selection will use the chronological validation period.

These settings are specified before fitting. They do not establish
calibrated probabilities or final model performance.

In [16]:
import sklearn

with experiment_path.open() as file:
    experiment = json.load(file)

model_settings = {
    "training_prior_reference": {
        "definition": "weighted_mean_of_training_labels",
    },
    "logistic_regression": {
        "preprocessing": "weighted_training_only_StandardScaler",
        "parameters": {
            "C": 1.0,
            "solver": "lbfgs",
            "l1_ratio": 0.0,
            "max_iter": 1000,
            "tol": 0.000001,
            "class_weight": None,
            "random_state": 42,
        },
    },
    "hist_gradient_boosting": {
        "preprocessing": "none",
        "parameters": {
            "loss": "log_loss",
            "learning_rate": 0.1,
            "max_iter": 100,
            "max_leaf_nodes": 15,
            "min_samples_leaf": 100,
            "l2_regularization": 1.0,
            "early_stopping": False,
            "class_weight": None,
            "random_state": 42,
        },
    },
}

weight_policy = {
    "fitting_weights": "inclusion_weight / mean_training_inclusion_weight",
    "scaler_weights": "same_as_fitting_weights",
    "prior_reference": "weighted_training_label_mean",
    "additional_class_balancing": False,
}

# Check that these settings can construct models in this environment.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

LogisticRegression(
    **model_settings["logistic_regression"]["parameters"]
)
HistGradientBoostingClassifier(
    **model_settings["hist_gradient_boosting"]["parameters"]
)

# Avoid silently changing settings if this cell is rerun later.
for key, proposed in [
    ("model_settings", model_settings),
    ("fitting_weight_policy", weight_policy),
]:
    if key in experiment and experiment[key] != proposed:
        raise ValueError(
            f"Existing {key} differs. Review before changing the experiment."
        )

experiment["model_settings"] = model_settings
experiment["fitting_weight_policy"] = weight_policy
experiment["development_environment"] = {
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
}
experiment["status"] = "model_settings_recorded_evaluation_details_pending"

# Replace the file only after the complete JSON has been written.
temporary_path = experiment_path.with_suffix(".json.tmp")
with temporary_path.open("w") as file:
    json.dump(experiment, file, indent=2)
    file.write("\n")
temporary_path.replace(experiment_path)

print("Scikit-learn version:", sklearn.__version__)
print("Model settings recorded.")
print("No models fitted yet.")

Scikit-learn version: 1.9.1
Model settings recorded.
No models fitted yet.


## 15. Evaluation and Selection Policy

Validation and test retain their natural eligible class balance.
Predictions will be processed in bounded batches.

I will accumulate positive and negative counts in fixed score bins.
Average precision calculated from these grouped scores is approximate
and will be labeled accordingly. Batch average-precision values will
not be averaged.

The model with the highest validation approximate average precision
will be selected. For that model, the alert threshold will maximize
validation F1 over the predefined score-bin boundaries. If thresholds
tie, prefer fewer alerts.

F1 gives precision and recall equal importance. It is an experimental
selection rule, not an operational safety requirement.

After selection, freeze the model and threshold before test evaluation.
Report precision, recall, confusion counts and alert burden alongside
approximate average precision. Calibration is not claimed.

Model alert thresholds and cascading-engine thresholds are separate.

In [17]:
with experiment_path.open() as file:
    experiment = json.load(file)

evaluation_details = {
    "batch_rows": 100000,
    "population": "all_eligible_rows_at_natural_prevalence",
    "evaluation_sample_weights": "none",
    "score_binning": {
        "method": "fixed_logarithmic_edges",
        "edges_definition": (
            "unique(concatenate([0.0], "
            "geomspace(1e-12, 1.0, 4096)))"
        ),
        "score_range": [0.0, 1.0],
        "nonfinite_or_out_of_range_scores": "error",
        "group_counts_across_batches_before_metrics": True,
    },
    "average_precision": {
        "method": "global_score_bin_counts",
        "reported_name": "approximate_average_precision",
        "limitation": (
            "Scores within each bin are treated as tied; "
            "this can differ from exact unbinned average precision."
        ),
        "average_batch_AP": False,
    },
    "model_selection": {
        "dataset": "validation",
        "criterion": "highest_approximate_average_precision",
        "exact_tie_preference": [
            "logistic_regression",
            "hist_gradient_boosting",
            "training_prior_reference",
        ],
    },
    "threshold_selection": {
        "dataset": "validation",
        "candidate_thresholds": "fixed_score_bin_edges",
        "decision_rule": "score >= threshold",
        "objective": "maximum_F1",
        "tie_break": "fewest_alerts_then_highest_threshold",
        "zero_denominator_metric_value": 0.0,
    },
    "final_test": {
        "use_frozen_selected_model": True,
        "use_frozen_validation_threshold": True,
        "retuning_allowed": False,
        "report": [
            "approximate_average_precision",
            "precision",
            "recall",
            "F1",
            "TP_FP_TN_FN",
            "total_alerts",
            "alerts_and_false_alerts_by_UTC_day",
            "borough_support_and_metrics",
        ],
    },
}

if (
    "evaluation_details" in experiment
    and experiment["evaluation_details"] != evaluation_details
):
    raise ValueError(
        "Existing evaluation settings differ. Review before changing them."
    )

experiment["evaluation_details"] = evaluation_details
experiment["status"] = "ready_for_prefit_git_checkpoint"

temporary_path = experiment_path.with_suffix(".json.tmp")
with temporary_path.open("w") as file:
    json.dump(experiment, file, indent=2)
    file.write("\n")
temporary_path.replace(experiment_path)

print("Evaluation and threshold-selection policy recorded.")
print("Model settings and evaluation plan are ready for Git review.")
print("No models fitted and no test predictions generated.")

Evaluation and threshold-selection policy recorded.
Model settings and evaluation plan are ready for Git review.
No models fitted and no test predictions generated.


## 16. Load and Validate the Training Sample

I will load the sampled training rows in a deterministic order and verify:

- Unique street-segment/hour keys.
- The eight required features are finite.
- Labels and inclusion weights match the experiment plan.
- All rows fall within the training interval.

Inclusion weights will be rescaled to mean 1 for fitting.
Validation and test data are not loaded in this step.

In [18]:
with experiment_path.open() as file:
    frozen_experiment = json.load(file)

FEATURES = frozen_experiment["features"]

# Feature names become SQL column identifiers.
if FEATURES != preparation_config["features"]:
    raise ValueError("Experiment and preparation feature orders differ.")

if not all(name.replace("_", "").isalnum() for name in FEATURES):
    raise ValueError("Unexpected feature column name.")

feature_sql = ", ".join(f'"{name}"' for name in FEATURES)

connection = duckdb.connect(str(database_path), read_only=True)

try:
    connection.execute("SET threads=2")
    connection.execute("SET memory_limit='2GB'")
    connection.execute("SET TimeZone='UTC'")

    training_data = connection.execute(
        f"""
        SELECT
            street_segment_id,
            prediction_time_utc::VARCHAR AS prediction_time_utc,
            {feature_sql},
            label,
            sample_weight
        FROM training_sample
        ORDER BY street_segment_id, prediction_time_utc
        """
    ).fetchdf()

finally:
    connection.close()

training_data["prediction_time_utc"] = pd.to_datetime(
    training_data["prediction_time_utc"], utc=True
)

sample_plan = frozen_experiment["training_sample"]

assert len(training_data) == sample_plan["actual_rows"]
assert training_data["label"].isin([0, 1]).all()
assert not training_data[
    ["street_segment_id", "prediction_time_utc"]
].isna().any().any()
assert not training_data.duplicated(
    ["street_segment_id", "prediction_time_utc"]
).any()

train_start, train_end = [
    pd.Timestamp(value)
    for value in frozen_experiment["splits"]["train"]
]

assert training_data["prediction_time_utc"].ge(train_start).all()
assert training_data["prediction_time_utc"].lt(train_end).all()

X_train = training_data[FEATURES].astype("float64").copy()
y_train = training_data["label"].to_numpy(dtype=np.int8)

assert np.isfinite(X_train.to_numpy()).all()

# Elevation may legitimately be below zero.
nonnegative_features = [
    feature for feature in FEATURES
    if feature != "elevation_mean_m"
]
assert X_train[nonnegative_features].ge(0).all().all()

rain_values = X_train[
    ["rain_1h_mm", "rain_3h_mm", "rain_6h_mm", "rain_24h_mm"]
].to_numpy()
assert (np.diff(rain_values, axis=1) >= -1e-9).all()

inclusion_weights = training_data["sample_weight"].to_numpy(
    dtype="float64"
)

expected_weights = np.where(
    y_train == 1,
    sample_plan["positive_inclusion_weight"],
    sample_plan["negative_inclusion_weight"],
)

assert np.array_equal(inclusion_weights, expected_weights)
assert int((y_train == 1).sum()) == sample_plan["positive_rows"]
assert int((y_train == 0).sum()) == sample_plan["negative_rows"]

weight_normalization_factor = float(inclusion_weights.mean())
fit_weights = inclusion_weights / weight_normalization_factor

training_prior = float(
    np.average(y_train, weights=inclusion_weights)
)

print(f"Training rows: {len(training_data):,}")
print("Feature matrix shape:", X_train.shape)
print("Feature order:", FEATURES)
print(f"Positive rows: {int(y_train.sum()):,}")
print(f"Weight normalization factor: {weight_normalization_factor:.6f}")
print(f"Mean fitting weight: {fit_weights.mean():.6f}")
print(f"Weighted training-prior score: {training_prior:.10f}")
print(
    "Loaded table memory:",
    f"{training_data.memory_usage(deep=True).sum() / 1024**2:.1f} MiB",
)
print("\nAll training-input checks passed. Ready to fit.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training rows: 666,857
Feature matrix shape: (666857, 8)
Feature order: ['elevation_mean_m', 'terrain_endpoint_grade_abs', 'terrain_relief_m', 'length_m', 'rain_1h_mm', 'rain_3h_mm', 'rain_6h_mm', 'rain_24h_mm']
Positive rows: 12,817
Weight normalization factor: 4903.919157
Mean fitting weight: 1.000000
Weighted training-prior score: 0.0000039193
Loaded table memory: 97.9 MiB

All training-input checks passed. Ready to fit.


## 17. Fit the Baseline Candidates

I will fit logistic regression and histogram gradient boosting using
the frozen settings and normalized inclusion weights.

The logistic regression scaler is fitted only on training data using
the same weights. The tree model uses the original feature values.
Computation is limited to two threads.

The weighted training prior is the constant-score reference.
No validation or test data is used during fitting.

Successful fitting does not establish prediction quality. Model
comparison and threshold selection follow on validation data.

In [19]:
import time
import warnings

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

settings = frozen_experiment["model_settings"]

logistic_model = Pipeline([
    ("scale", StandardScaler()),
    (
        "classifier",
        LogisticRegression(
            **settings["logistic_regression"]["parameters"]
        ),
    ),
])

tree_model = HistGradientBoostingClassifier(
    **settings["hist_gradient_boosting"]["parameters"]
)

fit_results = []
fit_warning_records = []

with threadpool_limits(limits=2):
    print("Fitting logistic regression...", flush=True)
    started = time.perf_counter()

    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")

        logistic_model.fit(
            X_train,
            y_train,
            scale__sample_weight=fit_weights,
            classifier__sample_weight=fit_weights,
        )

    logistic_convergence_warning = any(
        issubclass(item.category, ConvergenceWarning)
        for item in captured
    )

    for item in captured:
        fit_warning_records.append({
            "model": "logistic_regression",
            "category": item.category.__name__,
            "message": str(item.message),
        })

    fit_results.append({
        "model": "logistic_regression",
        "fit_seconds": time.perf_counter() - started,
        "iterations": int(
            logistic_model.named_steps["classifier"].n_iter_.max()
        ),
        "convergence_warning": logistic_convergence_warning,
    })

    print("Logistic regression fit finished.", flush=True)
    print("Fitting histogram gradient boosting...", flush=True)
    started = time.perf_counter()

    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        tree_model.fit(
            X_train,
            y_train,
            sample_weight=fit_weights,
        )

    for item in captured:
        fit_warning_records.append({
            "model": "hist_gradient_boosting",
            "category": item.category.__name__,
            "message": str(item.message),
        })

    fit_results.append({
        "model": "hist_gradient_boosting",
        "fit_seconds": time.perf_counter() - started,
        "iterations": int(tree_model.n_iter_),
        "convergence_warning": any(
            issubclass(item.category, ConvergenceWarning)
            for item in captured
        ),
    })

    print("Histogram gradient boosting fit finished.", flush=True)

candidate_models = {
    "logistic_regression": logistic_model,
    "hist_gradient_boosting": tree_model,
}

for name, model in candidate_models.items():
    if list(model.classes_) != [0, 1]:
        raise ValueError(f"Unexpected class order for {name}")

fit_summary = pd.DataFrame(fit_results)
display(fit_summary.round({"fit_seconds": 2}))

print("Constant training-prior reference:", training_prior)

if fit_warning_records:
    print("\nFit warnings — review before evaluation:")
    display(pd.DataFrame(fit_warning_records))
else:
    print("\nNo fit warnings were captured.")

if fit_summary["convergence_warning"].any():
    print("\nConvergence requires review before model comparison.")
else:
    print("\nBoth candidates fitted. Validation has not started.")

Fitting logistic regression...
Logistic regression fit finished.
Fitting histogram gradient boosting...
Histogram gradient boosting fit finished.


,model,fit_seconds,iterations,convergence_warning
0,logistic_regression,0.38,22,False
1,hist_gradient_boosting,2.39,100,False


Constant training-prior reference: 3.919316789834476e-06

No fit warnings were captured.

Both candidates fitted. Validation has not started.


## 18. Save Fitted Candidate Checkpoints

Both candidates fitted without captured warnings.

I will save the fitted models, experiment configuration, feature order,
class mapping, training summary and environment versions in a unique
run folder under data/processed/.

These checkpoints are not a final model release. Validation, threshold
selection and final test evaluation remain pending.

In [20]:
from datetime import datetime, timezone
import importlib.metadata
import joblib
import uuid

# Confirm the experiment file still matches the settings used for fitting.
with experiment_path.open() as file:
    current_experiment = json.load(file)

if current_experiment != frozen_experiment:
    raise ValueError(
        "Experiment settings changed after loading. "
        "Review before saving the fitted candidates."
    )

run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_"
    + uuid.uuid4().hex[:8]
)

run_directory = (
    PROCESSED / "climate_baseline_v2_runs" / run_id
)
run_directory.mkdir(parents=True, exist_ok=False)

model_files = {}

for name, model in candidate_models.items():
    model_path = run_directory / f"{name}.joblib"
    joblib.dump(model, model_path)

    model_files[name] = {
        "file": model_path.name,
        "sha256": sha256_file(model_path),
    }

with (run_directory / "experiment.json").open("x") as file:
    json.dump(frozen_experiment, file, indent=2)
    file.write("\n")

packages = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "joblib",
    "threadpoolctl",
    "duckdb",
]

checkpoint_manifest = {
    "run_id": run_id,
    "status": "candidates_fitted_not_validated",
    "experiment_id": frozen_experiment["experiment_id"],
    "experiment_sha256": sha256_file(
        run_directory / "experiment.json"
    ),
    "features": FEATURES,
    "classes": [0, 1],
    "training_rows": len(training_data),
    "training_positive_rows": int(y_train.sum()),
    "weight_normalization_factor": weight_normalization_factor,
    "training_prior_reference": training_prior,
    "fit_summary": fit_summary.to_dict(orient="records"),
    "fit_warnings": fit_warning_records,
    "model_files": model_files,
    "python_version": sys.version.split()[0],
    "package_versions": {
        package: importlib.metadata.version(package)
        for package in packages
    },
    "validation_performed": False,
    "test_performed": False,
    "final_model_selected": False,
    "score_calibration_claimed": False,
}

# Write this last: its presence marks a completed checkpoint.
manifest_path = run_directory / "checkpoint_manifest.json"
with manifest_path.open("x") as file:
    json.dump(checkpoint_manifest, file, indent=2, allow_nan=False)
    file.write("\n")

print("Candidate checkpoints saved to:")
print(run_directory)
print("\nModels:", list(model_files))
print("Validation and final model selection remain pending.")

Candidate checkpoints saved to:
/Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/data/processed/climate_baseline_v2_runs/20261024T194134Z_6afca461

Models: ['logistic_regression', 'hist_gradient_boosting']
Validation and final model selection remain pending.


## 19. Verify Fitted Models and Saved Checkpoints

I will inspect the fitted model dimensions, logistic coefficients and
prediction variation on a fixed sample of training inputs.

I will also reload each saved model, verify its file hash, and compare
its predictions with the in-memory model.

This is a technical check, not an evaluation of generalization.
Validation data has not been used yet.

In [21]:
from sklearn.utils.validation import check_is_fitted

# A fixed diagnostic sample, independent of labels.
rng = np.random.default_rng(42)
diagnostic_indices = np.sort(
    rng.choice(
        len(X_train),
        size=min(10000, len(X_train)),
        replace=False,
    )
)
X_diagnostic = X_train.iloc[diagnostic_indices]

print("Training matrix:", X_train.shape)
print("Diagnostic rows:", len(X_diagnostic))

# Inspect learned logistic parameters.
fitted_scaler = logistic_model.named_steps["scale"]
fitted_logistic = logistic_model.named_steps["classifier"]

check_is_fitted(fitted_scaler)
check_is_fitted(fitted_logistic)
check_is_fitted(tree_model)

assert fitted_logistic.n_features_in_ == len(FEATURES)
assert tree_model.n_features_in_ == len(FEATURES)

assert np.isfinite(fitted_logistic.coef_).all()
assert np.isfinite(fitted_logistic.intercept_).all()

coefficient_review = pd.DataFrame({
    "feature": FEATURES,
    "coefficient_after_scaling": fitted_logistic.coef_[0],
    "training_scaler_mean": fitted_scaler.mean_,
    "training_scaler_scale": fitted_scaler.scale_,
})

print("\nLogistic parameters:")
display(coefficient_review)
print("Logistic intercept:", fitted_logistic.intercept_[0])
print("Boosting iterations:", tree_model.n_iter_)

checkpoint_checks = []

with threadpool_limits(limits=2):
    for name, model in candidate_models.items():
        saved_info = checkpoint_manifest["model_files"][name]
        saved_path = run_directory / saved_info["file"]

        assert sha256_file(saved_path) == saved_info["sha256"], (
            f"Saved file hash differs for {name}"
        )

        # Load only our own locally generated, verified checkpoint.
        reloaded_model = joblib.load(saved_path)

        assert list(model.classes_) == [0, 1]
        assert list(reloaded_model.classes_) == [0, 1]

        original_scores = model.predict_proba(X_diagnostic)
        reloaded_scores = reloaded_model.predict_proba(X_diagnostic)

        assert np.isfinite(original_scores).all()
        assert (
            (original_scores >= 0) & (original_scores <= 1)
        ).all()
        assert np.allclose(
            original_scores.sum(axis=1), 1.0,
            rtol=0, atol=1e-10,
        )

        np.testing.assert_allclose(
            original_scores,
            reloaded_scores,
            rtol=0,
            atol=1e-12,
        )

        positive_scores = original_scores[:, 1]

        checkpoint_checks.append({
            "model": name,
            "minimum_score": positive_scores.min(),
            "median_score": np.median(positive_scores),
            "maximum_score": positive_scores.max(),
            "score_standard_deviation": positive_scores.std(),
            "distinct_scores_in_sample": np.unique(
                positive_scores
            ).size,
            "maximum_reload_difference": np.abs(
                original_scores - reloaded_scores
            ).max(),
        })

checkpoint_check_table = pd.DataFrame(checkpoint_checks)

print("\nPrediction and reload checks:")
display(checkpoint_check_table)

for row in checkpoint_checks:
    if row["distinct_scores_in_sample"] == 1:
        print(
            "REVIEW:",
            row["model"],
            "returned a constant score on this diagnostic sample."
        )

print("\nCheckpoint verification finished.")
print("These training-input checks are not validation metrics.")

Training matrix: (666857, 8)
Diagnostic rows: 10000

Logistic parameters:


,feature,coefficient_after_scaling,training_scaler_mean,training_scaler_scale
0,elevation_mean_m,-0.119754,18.990849,14.797627
1,terrain_endpoint_grade_abs,-0.093878,0.029410,0.046356
2,terrain_relief_m,0.030673,3.781301,4.411688
3,length_m,0.122724,121.607901,120.637339
4,rain_1h_mm,0.035580,0.125362,0.799305
5,rain_3h_mm,0.051869,0.378557,1.900628
6,rain_6h_mm,-0.002803,0.774448,3.320713
7,rain_24h_mm,0.248308,3.259099,8.899399


Logistic intercept: -12.59336474149921
Boosting iterations: 100

Prediction and reload checks:


,model,minimum_score,median_score,maximum_score,score_standard_deviation,distinct_scores_in_sample,maximum_reload_difference
0,logistic_regression,2.901889e-07,0.000003,0.009786,0.000117,9771,0.0
1,hist_gradient_boosting,1.047687e-06,0.000002,0.000070,0.000006,4887,0.0



Checkpoint verification finished.
These training-input checks are not validation metrics.


## 20. Validation Pipeline Smoke Check

Both fitted models produce varying scores, and their saved checkpoints
reproduce the in-memory predictions on the diagnostic sample.

I will test batched validation on the first configured validation hour.
This checks prediction, score binning and row accounting. It is not
enough data for model selection or threshold selection.

All three candidates use identical rows. Evaluation rows have no
sampling weights because their natural eligible prevalence is retained.

In [23]:
score_edges = np.unique(
    np.concatenate(([0.0], np.geomspace(1e-12, 1.0, 4096)))
)

def evaluate_validation_window(start, end):
    """Accumulate score-bin counts for all candidates on the same rows."""
    names = [
        "training_prior_reference",
        "logistic_regression",
        "hist_gradient_boosting",
    ]

    histograms = {
        name: {
            "positive": np.zeros(len(score_edges), dtype=np.int64),
            "negative": np.zeros(len(score_edges), dtype=np.int64),
        }
        for name in names
    }

    rows_seen = 0
    positives_seen = 0

    connection = duckdb.connect(str(database_path), read_only=True)

    try:
        connection.execute("SET threads=2")
        connection.execute("SET memory_limit='2GB'")
        connection.execute("SET TimeZone='UTC'")

        reader = connection.execute(
            f"""
            SELECT {feature_sql}, label
            FROM validation_full
            WHERE prediction_time_utc >= ?::TIMESTAMPTZ
              AND prediction_time_utc < ?::TIMESTAMPTZ
            """,
            [start.isoformat(), end.isoformat()],
        ).to_arrow_reader(
            batch_size=frozen_experiment[
                "evaluation_details"
            ]["batch_rows"]
        )

        with threadpool_limits(limits=2):
            for batch in reader:
                frame = batch.to_pandas()
                X_batch = frame[FEATURES].astype("float64")
                labels = frame["label"].to_numpy()

                if not np.isfinite(X_batch.to_numpy()).all():
                    raise ValueError("Nonfinite validation features.")

                if not np.isin(labels, [0, 1]).all():
                    raise ValueError("Unexpected validation label.")

                positive = labels == 1

                for name in names:
                    if name == "training_prior_reference":
                        scores = np.full(
                            len(frame), training_prior, dtype="float64"
                        )
                    else:
                        scores = candidate_models[name].predict_proba(
                            X_batch
                        )[:, 1]

                    if not (
                        np.isfinite(scores).all()
                        and (scores >= 0).all()
                        and (scores <= 1).all()
                    ):
                        raise ValueError(f"Invalid scores from {name}")

                    # Bins represent thresholds at their lower edges.
                    bins = np.searchsorted(
                        score_edges, scores, side="right"
                    ) - 1

                    histograms[name]["positive"] += np.bincount(
                        bins[positive], minlength=len(score_edges)
                    )
                    histograms[name]["negative"] += np.bincount(
                        bins[~positive], minlength=len(score_edges)
                    )

                rows_seen += len(frame)
                positives_seen += int(positive.sum())

    finally:
        connection.close()

    for name in names:
        assert histograms[name]["positive"].sum() == positives_seen
        assert (
            histograms[name]["positive"].sum()
            + histograms[name]["negative"].sum()
            == rows_seen
        )

    return histograms, rows_seen, positives_seen


validation_start = pd.Timestamp(
    frozen_experiment["splits"]["validation"][0]
)
smoke_end = validation_start + pd.Timedelta(hours=1)

started = time.perf_counter()

smoke_histograms, smoke_rows, smoke_positives = (
    evaluate_validation_window(validation_start, smoke_end)
)

if smoke_rows == 0:
    raise ValueError(
        "The selected hour has no eligible rows. "
        "Review coverage before choosing another smoke window."
    )

print("Window:", validation_start, "to", smoke_end)
print(f"Eligible rows processed: {smoke_rows:,}")
print(f"Positive rows: {smoke_positives:,}")
print(f"Elapsed seconds: {time.perf_counter() - started:.2f}")
print("All three candidates have matching row and label counts.")
print("Smoke check only — no model or threshold selected.")

Window: 2025-01-01 05:00:00+00:00 to 2025-01-01 06:00:00+00:00
Eligible rows processed: 91,895
Positive rows: 0
Elapsed seconds: 0.33
All three candidates have matching row and label counts.
Smoke check only — no model or threshold selected.


## 21. Begin Validation with Saved Daily Progress

I will process the first 24-hour validation window and save its score-bin
counts, row totals and provenance.

This measures processing speed and preserves completed work. Model and
threshold selection will use the combined full validation population.

In [24]:
validation_directory = run_directory / "validation"
validation_directory.mkdir(exist_ok=True)

first_day_start = validation_start
first_day_end = min(
    first_day_start + pd.Timedelta(days=1),
    pd.Timestamp(frozen_experiment["splits"]["validation"][1]),
)

window_id = first_day_start.strftime("%Y%m%dT%H%M%SZ")
histogram_path = validation_directory / f"{window_id}.npz"
metadata_path = validation_directory / f"{window_id}.json"

if histogram_path.exists() or metadata_path.exists():
    raise FileExistsError(
        "This window already has saved output. "
        "Do not overwrite it; send me the message so we can check it."
    )

started = time.perf_counter()

day_histograms, day_rows, day_positives = evaluate_validation_window(
    first_day_start, first_day_end
)

elapsed = time.perf_counter() - started

arrays = {"score_edges": score_edges}

for name, histogram in day_histograms.items():
    arrays[f"{name}__positive"] = histogram["positive"]
    arrays[f"{name}__negative"] = histogram["negative"]

temporary_histogram = histogram_path.with_suffix(".npz.tmp")

with temporary_histogram.open("wb") as file:
    np.savez_compressed(file, **arrays)

temporary_histogram.replace(histogram_path)

day_metadata = {
    "run_id": run_id,
    "start_utc": first_day_start.isoformat(),
    "end_utc_exclusive": first_day_end.isoformat(),
    "rows": day_rows,
    "positive_rows": day_positives,
    "negative_rows": day_rows - day_positives,
    "elapsed_seconds": elapsed,
    "histogram_file": histogram_path.name,
    "histogram_sha256": sha256_file(histogram_path),
    "experiment_sha256": checkpoint_manifest["experiment_sha256"],
    "model_files": checkpoint_manifest["model_files"],
}

# Write the completion record only after saving the histogram.
with metadata_path.open("x") as file:
    json.dump(day_metadata, file, indent=2, allow_nan=False)
    file.write("\n")

print("Completed:", first_day_start, "to", first_day_end)
print(f"Rows: {day_rows:,}")
print(f"Positives: {day_positives:,}")
print(f"Elapsed: {elapsed:.2f} seconds")
print("Progress saved:", metadata_path)
print("No model or threshold selected.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Completed: 2025-01-01 05:00:00+00:00 to 2025-01-02 05:00:00+00:00
Rows: 2,205,480
Positives: 6
Elapsed: 3.31 seconds
Progress saved: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/data/processed/climate_baseline_v2_runs/20261024T194134Z_6afca461/validation/20250101T050000Z.json
No model or threshold selected.


## 22. Complete Batched Validation

I will evaluate all remaining validation windows and save their score-bin
counts. Completed windows are reused only when their provenance, hashes,
bin definitions and row counts match.

Each candidate sees the same eligible rows. No test data is evaluated.
Model and threshold selection follow after all validation totals reconcile.

In [25]:
validation_end = pd.Timestamp(
    frozen_experiment["splits"]["validation"][1]
)

candidate_names = [
    "training_prior_reference",
    "logistic_regression",
    "hist_gradient_boosting",
]

# Check saved experiment/model files before continuing.
assert sha256_file(run_directory / "experiment.json") == (
    checkpoint_manifest["experiment_sha256"]
)

with experiment_path.open() as file:
    assert json.load(file) == frozen_experiment

for info in checkpoint_manifest["model_files"].values():
    assert sha256_file(run_directory / info["file"]) == info["sha256"]

windows = []
start = validation_start

while start < validation_end:
    end = min(start + pd.Timedelta(days=1), validation_end)
    windows.append((start, end))
    start = end

validation_records = []
started_all = time.perf_counter()

print(f"Checking/processing {len(windows)} validation windows...", flush=True)

for number, (start, end) in enumerate(windows, start=1):
    window_id = start.strftime("%Y%m%dT%H%M%SZ")
    histogram_path = validation_directory / f"{window_id}.npz"
    metadata_path = validation_directory / f"{window_id}.json"

    if metadata_path.exists():
        with metadata_path.open() as file:
            metadata = json.load(file)

        assert metadata["run_id"] == run_id
        assert metadata["start_utc"] == start.isoformat()
        assert metadata["end_utc_exclusive"] == end.isoformat()
        assert metadata["experiment_sha256"] == (
            checkpoint_manifest["experiment_sha256"]
        )
        assert metadata["model_files"] == checkpoint_manifest["model_files"]
        assert metadata["histogram_file"] == histogram_path.name
        assert histogram_path.is_file()
        assert sha256_file(histogram_path) == metadata["histogram_sha256"]

    else:
        if histogram_path.exists():
            raise FileExistsError(
                f"Incomplete window found: {histogram_path}. "
                "Review it before continuing."
            )

        started = time.perf_counter()
        histograms, rows, positives = evaluate_validation_window(start, end)

        arrays = {"score_edges": score_edges}
        for name, histogram in histograms.items():
            arrays[f"{name}__positive"] = histogram["positive"]
            arrays[f"{name}__negative"] = histogram["negative"]

        temporary_path = histogram_path.with_suffix(".npz.tmp")
        with temporary_path.open("wb") as file:
            np.savez_compressed(file, **arrays)
        temporary_path.replace(histogram_path)

        metadata = {
            "run_id": run_id,
            "start_utc": start.isoformat(),
            "end_utc_exclusive": end.isoformat(),
            "rows": rows,
            "positive_rows": positives,
            "negative_rows": rows - positives,
            "elapsed_seconds": time.perf_counter() - started,
            "histogram_file": histogram_path.name,
            "histogram_sha256": sha256_file(histogram_path),
            "experiment_sha256": checkpoint_manifest["experiment_sha256"],
            "model_files": checkpoint_manifest["model_files"],
        }

        temporary_metadata = metadata_path.with_suffix(".json.tmp")
        with temporary_metadata.open("w") as file:
            json.dump(metadata, file, indent=2, allow_nan=False)
            file.write("\n")
        temporary_metadata.replace(metadata_path)

    # Verify saved histogram contents for both new and reused windows.
    with np.load(histogram_path, allow_pickle=False) as saved:
        assert np.array_equal(saved["score_edges"], score_edges)

        for name in candidate_names:
            positive_bins = saved[f"{name}__positive"]
            negative_bins = saved[f"{name}__negative"]

            assert positive_bins.shape == score_edges.shape
            assert negative_bins.shape == score_edges.shape
            assert np.issubdtype(positive_bins.dtype, np.integer)
            assert np.issubdtype(negative_bins.dtype, np.integer)
            assert (positive_bins >= 0).all()
            assert (negative_bins >= 0).all()

            assert int(positive_bins.sum()) == metadata["positive_rows"]
            assert int(negative_bins.sum()) == metadata["negative_rows"]

    assert metadata["positive_rows"] + metadata["negative_rows"] == (
        metadata["rows"]
    )

    validation_records.append(metadata)

    if number % 10 == 0 or number == len(windows):
        processed_rows = sum(item["rows"] for item in validation_records)
        print(
            f"{number}/{len(windows)} windows verified; "
            f"{processed_rows:,} rows accounted for",
            flush=True,
        )

validation_progress = pd.DataFrame(validation_records)

expected_validation = counts.loc[counts["split"] == "validation"]
expected_rows = int(expected_validation["rows"].sum())
expected_positives = int(
    expected_validation.loc[
        expected_validation["label"] == 1, "rows"
    ].sum()
)

assert int(validation_progress["rows"].sum()) == expected_rows
assert int(validation_progress["positive_rows"].sum()) == expected_positives

print(f"\nValidation rows: {expected_rows:,}")
print(f"Validation positives: {expected_positives:,}")
print(f"This session: {(time.perf_counter() - started_all) / 60:.1f} minutes")
print("All validation windows completed and totals reconciled.")
print("No model or threshold selected. Test data remains unevaluated.")

Checking/processing 358 validation windows...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (384 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

358/358 windows verified; 782,119,322 rows accounted for

Validation rows: 782,119,322
Validation positives: 3,383
This session: 20.1 minutes
All validation windows completed and totals reconciled.
No model or threshold selected. Test data remains unevaluated.


## 23. Compare Candidates on Full Validation

I will combine score-bin counts across all completed validation windows
before calculating metrics.

Average precision is approximate because scores within each bin are
treated as tied. Threshold confusion counts are calculated at the fixed
bin boundaries.

The candidate with the highest validation approximate average precision
is selected. Its threshold maximizes validation F1, with ties resolved
by fewer alerts and then the higher threshold.

These are validation results used for selection, not final test results.

In [26]:
combined_histograms = {
    name: {
        "positive": np.zeros(len(score_edges), dtype=np.int64),
        "negative": np.zeros(len(score_edges), dtype=np.int64),
    }
    for name in candidate_names
}

for metadata in validation_records:
    path = validation_directory / metadata["histogram_file"]

    if sha256_file(path) != metadata["histogram_sha256"]:
        raise ValueError(f"Histogram changed: {path.name}")

    with np.load(path, allow_pickle=False) as saved:
        assert np.array_equal(saved["score_edges"], score_edges)

        for name in candidate_names:
            for label_name in ["positive", "negative"]:
                combined_histograms[name][label_name] += saved[
                    f"{name}__{label_name}"
                ]

validation_curves = {}
comparison_rows = []

validation_days = (
    validation_end - validation_start
).total_seconds() / 86400

for name in candidate_names:
    # Descending thresholds: highest scores enter first.
    positive_bins = combined_histograms[name]["positive"][::-1]
    negative_bins = combined_histograms[name]["negative"][::-1]
    thresholds = score_edges[::-1]

    total_positive = int(positive_bins.sum())
    total_negative = int(negative_bins.sum())

    assert total_positive == expected_positives
    assert total_positive + total_negative == expected_rows

    true_positive = np.cumsum(positive_bins)
    false_positive = np.cumsum(negative_bins)
    alerts = true_positive + false_positive

    precision = np.divide(
        true_positive,
        alerts,
        out=np.zeros(len(alerts), dtype=float),
        where=alerts > 0,
    )
    recall = true_positive / total_positive

    f1 = np.divide(
        2.0 * true_positive,
        total_positive + alerts,
        out=np.zeros(len(alerts), dtype=float),
        where=(total_positive + alerts) > 0,
    )

    # Global grouped-score AP, not an average of window AP values.
    approximate_ap = float(
        np.sum((positive_bins / total_positive) * precision)
    )

    curve = pd.DataFrame({
        "threshold": thresholds,
        "precision": precision,
        "recall": recall,
        "F1": f1,
        "TP": true_positive,
        "FP": false_positive,
        "FN": total_positive - true_positive,
        "TN": total_negative - false_positive,
        "alerts": alerts,
    })

    validation_curves[name] = curve

    best = (
        curve.loc[curve["F1"] == curve["F1"].max()]
        .sort_values(
            ["alerts", "threshold"],
            ascending=[True, False],
        )
        .iloc[0]
    )

    comparison_rows.append({
        "model": name,
        "approximate_AP": approximate_ap,
        "validation_F1_threshold": float(best["threshold"]),
        "precision": float(best["precision"]),
        "recall": float(best["recall"]),
        "F1": float(best["F1"]),
        "TP": int(best["TP"]),
        "FP": int(best["FP"]),
        "FN": int(best["FN"]),
        "TN": int(best["TN"]),
        "alerts": int(best["alerts"]),
        "mean_alerts_per_24h": float(best["alerts"]) / validation_days,
        "mean_false_alerts_per_24h": float(best["FP"]) / validation_days,
    })

validation_comparison = pd.DataFrame(comparison_rows)

tie_preference = frozen_experiment[
    "evaluation_details"
]["model_selection"]["exact_tie_preference"]

preference_rank = {
    name: rank for rank, name in enumerate(tie_preference)
}

validation_comparison["_tie_rank"] = (
    validation_comparison["model"].map(preference_rank)
)

validation_comparison = (
    validation_comparison.sort_values(
        ["approximate_AP", "_tie_rank"],
        ascending=[False, True],
    )
    .drop(columns="_tie_rank")
    .reset_index(drop=True)
)

selected_validation_result = validation_comparison.iloc[0]
selected_model_name = selected_validation_result["model"]
selected_threshold = float(
    selected_validation_result["validation_F1_threshold"]
)

print(
    "Validation positive prevalence:",
    f"{expected_positives / expected_rows:.10f}",
)

display(validation_comparison)

print("\nSelected by the predefined validation policy:")
print("Candidate:", selected_model_name)
print("Threshold:", selected_threshold)
print("Score calibration has not been established.")
print("Final test evaluation has not started.")

Validation positive prevalence: 0.0000043254


,model,approximate_AP,validation_F1_threshold,precision,recall,F1,TP,FP,FN,TN,alerts,mean_alerts_per_24h,mean_false_alerts_per_24h
0,hist_gradient_boosting,0.000035,0.000069,0.000138,0.005912,0.000269,20,145219,3363,781970720,145239,4.056955e+02,4.056397e+02
1,logistic_regression,0.000021,0.000890,0.000352,0.000296,0.000321,1,2840,3382,782113099,2841,7.935754e+00,7.932961e+00
2,training_prior_reference,0.000004,0.000004,0.000004,1.000000,0.000009,3383,782115939,0,0,782119322,2.184691e+06,2.184681e+06



Selected by the predefined validation policy:
Candidate: hist_gradient_boosting
Threshold: 6.94637526817251e-05
Score calibration has not been established.
Final test evaluation has not started.


## 24. Diagnose Training Versus Validation Performance

The first experiment has weak validation performance. Before changing
settings, I will inspect performance on the training sample.

Training metrics use inverse inclusion weights to estimate performance
on the eligible training population. They are approximate population
estimates and are optimistic because these rows were used for fitting.

Average precision uses the same score bins as validation.
Operating thresholds remain those already selected on validation.
No model is refitted and no test data is inspected.

In [27]:
# Preserve the first experiment's validation comparison.
comparison_path = run_directory / "validation_comparison.csv"
comparison_text = validation_comparison.to_csv(index=False)

if comparison_path.exists():
    if comparison_path.read_text() != comparison_text:
        raise ValueError(
            "Saved validation results differ. Review before overwriting."
        )
else:
    with comparison_path.open("x") as file:
        file.write(comparison_text)

diagnostic_rows = []
batch_size = frozen_experiment["evaluation_details"]["batch_rows"]

with threadpool_limits(limits=2):
    for name in candidate_names:
        print("Checking training performance:", name, flush=True)

        weighted_positive_bins = np.zeros(
            len(score_edges), dtype="float64"
        )
        weighted_negative_bins = np.zeros(
            len(score_edges), dtype="float64"
        )

        threshold = float(
            validation_comparison.loc[
                validation_comparison["model"] == name,
                "validation_F1_threshold",
            ].iloc[0]
        )

        weighted_tp = 0.0
        weighted_fp = 0.0

        for start in range(0, len(X_train), batch_size):
            stop = min(start + batch_size, len(X_train))
            X_batch = X_train.iloc[start:stop]
            labels = y_train[start:stop]
            weights = inclusion_weights[start:stop]

            if name == "training_prior_reference":
                scores = np.full(len(X_batch), training_prior)
            else:
                scores = candidate_models[name].predict_proba(
                    X_batch
                )[:, 1]

            if not (
                np.isfinite(scores).all()
                and (scores >= 0).all()
                and (scores <= 1).all()
            ):
                raise ValueError(f"Invalid training scores: {name}")

            positive = labels == 1
            bins = np.searchsorted(
                score_edges, scores, side="right"
            ) - 1

            weighted_positive_bins += np.bincount(
                bins[positive],
                weights=weights[positive],
                minlength=len(score_edges),
            )
            weighted_negative_bins += np.bincount(
                bins[~positive],
                weights=weights[~positive],
                minlength=len(score_edges),
            )

            alerts = scores >= threshold
            weighted_tp += weights[alerts & positive].sum()
            weighted_fp += weights[alerts & ~positive].sum()

        positive_bins = weighted_positive_bins[::-1]
        negative_bins = weighted_negative_bins[::-1]

        cumulative_tp = np.cumsum(positive_bins)
        cumulative_fp = np.cumsum(negative_bins)

        precision_curve = np.divide(
            cumulative_tp,
            cumulative_tp + cumulative_fp,
            out=np.zeros_like(cumulative_tp),
            where=(cumulative_tp + cumulative_fp) > 0,
        )

        total_positive_weight = positive_bins.sum()

        training_ap = float(
            np.sum(
                positive_bins
                / total_positive_weight
                * precision_curve
            )
        )

        training_precision = (
            weighted_tp / (weighted_tp + weighted_fp)
            if weighted_tp + weighted_fp > 0 else 0.0
        )
        training_recall = weighted_tp / total_positive_weight

        validation_row = validation_comparison.loc[
            validation_comparison["model"] == name
        ].iloc[0]

        diagnostic_rows.append({
            "model": name,
            "weighted_training_approx_AP": training_ap,
            "validation_approx_AP": float(
                validation_row["approximate_AP"]
            ),
            "validation_selected_threshold": threshold,
            "training_precision_estimate": training_precision,
            "validation_precision": float(
                validation_row["precision"]
            ),
            "training_recall": training_recall,
            "validation_recall": float(validation_row["recall"]),
        })

training_validation_diagnosis = pd.DataFrame(diagnostic_rows)
display(training_validation_diagnosis)

print("First validation comparison saved:", comparison_path)
print("No models changed. No test data evaluated.")

Checking training performance: training_prior_reference
Checking training performance: logistic_regression
Checking training performance: hist_gradient_boosting


,model,weighted_training_approx_AP,validation_approx_AP,validation_selected_threshold,training_precision_estimate,validation_precision,training_recall,validation_recall
0,training_prior_reference,0.000004,0.000004,0.000004,0.000004,0.000004,1.000000,1.000000
1,logistic_regression,0.000630,0.000021,0.000890,0.001180,0.000352,0.009675,0.000296
2,hist_gradient_boosting,0.000051,0.000035,0.000069,0.000278,0.000138,0.046267,0.005912


First validation comparison saved: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/data/processed/climate_baseline_v2_runs/20261024T194134Z_6afca461/validation_comparison.csv
No models changed. No test data evaluated.


## 25. Compare Scores on Training and Validation Positives

I will compare score distributions for eligible positive rows in training
and validation.

This checks whether positive examples receive lower scores in the later
period. It does not explain the cause by itself and does not measure
false-alert burden.

The original models remain unchanged. Test data remains untouched.

In [28]:
connection = duckdb.connect(str(database_path), read_only=True)

try:
    connection.execute("SET threads=2")
    connection.execute("SET memory_limit='2GB'")

    validation_positives = connection.execute(
        f"""
        SELECT {feature_sql}
        FROM validation_full
        WHERE label = 1
        """
    ).fetchdf()

finally:
    connection.close()

assert len(validation_positives) == expected_positives

X_validation_positive = validation_positives[FEATURES].astype("float64")
X_training_positive = X_train.loc[y_train == 1]

assert np.isfinite(X_validation_positive.to_numpy()).all()

score_distribution_rows = []

with threadpool_limits(limits=2):
    for name, model in candidate_models.items():
        for population, inputs in [
            ("training_positive", X_training_positive),
            ("validation_positive", X_validation_positive),
        ]:
            scores = model.predict_proba(inputs)[:, 1]

            quantiles = np.quantile(
                scores, [0.0, 0.25, 0.5, 0.75, 0.95, 1.0]
            )

            score_distribution_rows.append({
                "model": name,
                "population": population,
                "rows": len(scores),
                "minimum": quantiles[0],
                "p25": quantiles[1],
                "median": quantiles[2],
                "p75": quantiles[3],
                "p95": quantiles[4],
                "maximum": quantiles[5],
            })

positive_score_review = pd.DataFrame(score_distribution_rows)
display(positive_score_review)

print(
    "Total normalized positive training weight:",
    float(fit_weights[y_train == 1].sum()),
)
print(
    "Total normalized negative training weight:",
    float(fit_weights[y_train == 0].sum()),
)

print("Original models unchanged; test data remains untouched.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,model,population,rows,minimum,p25,median,p75,p95,maximum
0,logistic_regression,training_positive,12817,2.901889e-07,0.000003,0.000004,0.000008,0.000064,0.219112
1,logistic_regression,validation_positive,3383,8.473781e-07,0.000003,0.000004,0.000006,0.000014,0.000894
2,hist_gradient_boosting,training_positive,12817,1.047687e-06,0.000003,0.000007,0.000022,0.000068,0.000070
3,hist_gradient_boosting,validation_positive,3383,1.047687e-06,0.000003,0.000007,0.000019,0.000039,0.000070


Total normalized positive training weight: 2.6136238365186495
Total normalized negative training weight: 666854.3863761635
Original models unchanged; test data remains untouched.


## 26. Controlled Tuning Experiment: Tree Weight Scale

The initial candidates performed poorly on validation. This follow-up
experiment is informed by those validation results.

I will test histogram gradient boosting with original inclusion weights:
1 for positives and 5,000 for sampled negatives. All other tree settings,
features, sampling and split boundaries remain unchanged.

This tests weight scale together with its interaction with regularization
and internal fitting constraints. It does not change relative class weights
or add class balancing.

The original experiment and checkpoints are preserved. Validation is now
a reused development set; the final test remains untouched.